# 항 중요도 분석 - 경로별 기여도 측정

- Tutorial ID: `adv-5-1`
- Tutorial: 항 중요도 분석
- Section ID: `adv-5-1-1`
- Section: 경로별 기여도 측정


In [ ]:
# ============================================================
# 코드 읽는 법 — 경로별 기여도 측정
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("항 중요도 분석")
print("=" * 60)

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

V = 8
d_model = 12
d_head = 6
n_heads = 2
np.random.seed(42)

W_E = np.random.randn(V, d_model) * 0.3
W_U = np.random.randn(d_model, V) * 0.2

# 2개 레이어, 각 2개 헤드
W_Q = [[np.random.randn(d_model, d_head) * 0.2 for _ in range(n_heads)] for _ in range(2)]
W_K = [[np.random.randn(d_model, d_head) * 0.2 for _ in range(n_heads)] for _ in range(2)]
W_V = [[np.random.randn(d_model, d_head) * 0.2 for _ in range(n_heads)] for _ in range(2)]
W_O = [[np.random.randn(d_head, d_model) * 0.2 for _ in range(n_heads)] for _ in range(2)]

seq = [0, 1, 2, 3]
X = W_E[seq]
mask = np.triu(np.full((4, 4), -1e9), k=1)

def attn_head(X, wq, wk, wv, wo, mask):
    Q, K, V = X @ wq, X @ wk, X @ wv
        A = softmax(Q @ K.T / np.sqrt(d_head) + mask)
    return (A @ V) @ wo, A

# 전방 패스
h0_outs = []
for h in range(n_heads):
    out, _ = attn_head(X, W_Q[0][h], W_K[0][h], W_V[0][h], W_O[0][h], mask)
    h0_outs.append(out)
X1 = X + sum(h0_outs)

h1_outs = []
for h in range(n_heads):
    out, _ = attn_head(X1, W_Q[1][h], W_K[1][h], W_V[1][h], W_O[1][h], mask)
    h1_outs.append(out)
X2 = X1 + sum(h1_outs)

logits = X2[-1] @ W_U

# 기여도 분해
direct = X[-1] @ W_U
print("\n1. 직접 로짓 기여 (DLA)")
print(f"   직접 경로: ||{np.linalg.norm(direct):.4f}||")

for l in range(2):
    outs = h0_outs if l == 0 else h1_outs
        for h in range(n_heads):
        contrib = outs[h][-1] @ W_U
        print(f"   L{l}H{h}: ||{np.linalg.norm(contrib):.4f}||")

# 합성 강도
print("\n2. 합성 강도 (K-합성)")
for h1 in range(n_heads):
    QK1 = W_Q[1][h1] @ W_K[1][h1].T
        for h0 in range(n_heads):
        OV0 = W_V[0][h0] @ W_O[0][h0]
        AB = QK1.T @ OV0
        score = np.linalg.norm(AB) / (np.linalg.norm(QK1) * np.linalg.norm(OV0) + 1e-8)
        print(f"   L1H{h1}←L0H{h0}: {score:.4f}")

expected = 1 / np.sqrt(d_head)
print(f"   랜덤 기대값: {expected:.4f}")

# 어블레이션
print("\n3. 어블레이션 분석")
base_pred = np.argmax(logits)
base_conf = softmax(logits)[base_pred]
print(f"   기준: T{base_pred} ({base_conf:.3f})")

for name, term in [("직접", direct)] + [(f"L0H{h}", h0_outs[h][-1] @ W_U) for h in range(n_heads)] + [(f"L1H{h}", h1_outs[h][-1] @ W_U) for h in range(n_heads)]:
    ablated = logits - term
    conf = softmax(ablated)[base_pred]
    delta = abs(base_conf - conf)
    bar = '█' * int(delta * 50)
    print(f"   {name:6s} 제거: Δ={delta:.4f} {bar}")